# Day 1 · Module 5: Agent Teams (Peers, Personalities, and the Price of Debate)

**Objective:** run Claude Code's experimental **Agent Teams** feature on a real ticket — watch teammates argue with each other, bend their personalities, then run the *same ticket* with plain subagents and let your own token numbers decide when a team is worth it.


## Relevant Claude Code Docs
Review these before starting the module:
- [Orchestrate teams of Claude Code sessions](https://code.claude.com/docs/en/agent-teams)
- [Manage multiple agents with agent view](https://code.claude.com/docs/en/agent-view)
- [Run agents in parallel](https://code.claude.com/docs/en/agents)

## 1 · The idea

Module 3's subagents are **workers**: each gets its own context window, does one focused job, and reports back to the one agent that spawned it. All coordination flows through that single hub — no worker ever talks to another worker.

An **agent team** is different in kind, not degree. Each teammate is a *full Claude Code session*: its own context window, its own tool loop, its own turn-taking. Teammates coordinate through two shared surfaces — a **task list** they claim work from, and a **mailbox** they use to message *each other* directly, not just the lead. The session you started becomes the **team lead**: it spawns teammates, watches their progress, and gets notified when they idle or fail.

| | Subagents (Module 3) | Agent team (this module) |
|---|---|---|
| What each one is | a task run in its own context | a full peer session |
| Talks to | only the agent that spawned it | any teammate, directly |
| Coordination | hub-and-spoke; caller owns all state | shared task list + peer mailboxes |
| Result handling | summarized back into one context | each peer keeps its own context |
| Token cost | low — one main context persists | high — N full sessions run in parallel |

The one-line test: if the work is *decomposable* (map this, test that, report back), subagents fit. If the work needs *argument* — competing hypotheses, adversarial review, someone changing their mind because a peer pushed back — that's what teams are for, because a subagent cannot talk to another subagent.


### Why teams exist — and what they cost

The honest framing first: this feature is **experimental** (off by default, breaking changes between releases) and **expensive**. Anthropic's own docs measure an agent team at roughly **7× the tokens** of a single session doing comparable work in plan mode, and their engineering write-up on multi-agent research systems reports multi-agent setups burning ~15× the tokens of a plain chat. Every teammate is a separate context window that stays warm until it's shut down.

So why pay it? Because some failure modes only surface under *disagreement*. One agent holding every point of view argues with itself silently and approves its own work. A team makes the argument externally visible — and steerable: you can open one teammate's transcript mid-debate and push on it.

This module doesn't ask you to take either claim on faith. You'll run the same ticket both ways and read the bill.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` throughout.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


### Preflight: is the feature available to you?

Agent Teams needs Claude Code **v2.1.32 or newer** and the env flag `CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1`. The flag already ships in this repo's `.claude/settings.json`, so any `claude` session started from the repo root picks it up. The cell below checks both. **If it fails**, do Parts A and B from the recorded run in `reference/m5/sample-run/` (or watch the facilitator demo) — Parts C and D work for everyone regardless.


In [ ]:
import json, pathlib, re, subprocess
ok = True
v = subprocess.run(["claude", "--version"], capture_output=True, text=True)
raw = (v.stdout or v.stderr).strip()
m = re.search(r"(\d+)\.(\d+)\.(\d+)", raw)
if not m:
    ok = False
    print("[ ] could not run `claude --version` — is Claude Code on your PATH?")
else:
    ver = tuple(int(x) for x in m.groups())
    good = ver >= (2, 1, 32)
    ok &= good
    print(f"[{'x' if good else ' '}] Claude Code {'.'.join(map(str, ver))} (need >= 2.1.32)")
settings = pathlib.Path(proj) / ".claude" / "settings.json"
flag = json.loads(settings.read_text()).get("env", {}).get("CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS") == "1"
ok &= flag
print(f"[{'x' if flag else ' '}] CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1 in .claude/settings.json")
agents = [p.name for p in (pathlib.Path(proj) / ".claude" / "agents").glob("*.md")]
for name in ("shipit.md", "skeptic.md", "merchant.md"):
    present = name in agents
    ok &= present
    print(f"[{'x' if present else ' '}] teammate definition .claude/agents/{name}")
print()
print("Preflight PASSED — run Parts A-D live." if ok else
      "Preflight FAILED — use reference/m5/sample-run/ for Parts A-B; Parts C-D still run live.")


### Mechanics: the team is files on disk

Two things make Agent Teams observable rather than magic:

- **The shared task list** lives at `~/.claude/tasks/{team-name}/` — pending / in-progress / completed, with dependencies. It persists across resumed sessions.
- **The mailboxes** live at `~/.claude/teams/{team-name}/inboxes/{agent-name}.json` — every peer-to-peer message is a JSON record you can open. (The `teams/` directory is cleaned up when the session ends, which is why the exercise has you snapshot it *while the team is running*.)

The team name is usually `session-` plus the first 8 characters of the lead's session ID; you may also see a team named `default`.

In the default **in-process** display, teammates appear in an agent panel under your prompt:

| Key | Does |
|---|---|
| Up / Down | select a teammate |
| Enter | open the selected teammate's transcript — type to message it directly |
| Esc | interrupt the selected teammate's turn |
| Ctrl+T | toggle the shared task list |
| x | shut down the selected teammate |

(Split-pane display exists for tmux/iTerm2, but it doesn't work in the VS Code integrated terminal — this workshop uses in-process mode everywhere.)

**Personalities.** A teammate can be spawned from any subagent definition in `.claude/agents/` — "spawn a teammate using the `skeptic` agent type" appends that file's body to the teammate's system prompt and honors its `tools:` allowlist. Three deliberately clashing charters ship with this repo:

- **`shipit`** — Ship-It Implementer. Smallest diff that closes the ticket; biased toward wiring `webhooks.deliver()` straight into `posting.post()`. The only teammate allowed to edit code.
- **`skeptic`** — Reliability Skeptic. Attacks every proposal: slow endpoint, failure, retry, idempotency. Must cite files.
- **`merchant`** — Merchant Advocate. Argues from the subscriber's ledger: a lost `transfer.posted` is unshipped goods; a duplicate is double fulfillment.

Each ends with the same house rule: *disagree in the open — send the objection to the teammate you disagree with, by name, then summarize for the lead.* That rule is what turns the mailbox into a debate you can watch.

Open all three now (`.claude/agents/shipit.md`, `skeptic.md`, `merchant.md`) — the exercise asks you to predict the argument before you see it.


### Grounding: read the coordination surfaces yourself

Run this **while a team session is live** (after your first spawn in Part A — it's safe to run now too; it just tells you no team exists yet). It finds your newest team and prints the task list and real inbox traffic. This is the grounding moment: the "chat" is not UI theater, it's JSON you can open.


In [ ]:
import json, pathlib

def latest_team():
    # A live team always has a dir under ~/.claude/teams (often "session-<id>",
    # sometimes "default"); prefer that over ~/.claude/tasks, which can hold
    # unrelated task lists.
    for base in (pathlib.Path.home() / ".claude" / "teams",
                 pathlib.Path.home() / ".claude" / "tasks"):
        dirs = [d for d in base.iterdir() if d.is_dir()] if base.exists() else []
        if dirs:
            return max(dirs, key=lambda d: d.stat().st_mtime).name
    return None

team = latest_team()
if not team:
    print("No team found yet — come back after your first spawn in Part A.")
else:
    print("newest team:", team)
    tasks = pathlib.Path.home() / ".claude" / "tasks" / team
    if tasks.exists():
        print("\n== shared task list ==")
        for f in sorted(tasks.rglob("*"))[:20]:
            if f.is_file():
                print(" ", f.relative_to(tasks))
    inboxes = pathlib.Path.home() / ".claude" / "teams" / team / "inboxes"
    if inboxes.exists():
        print("\n== mailbox traffic (first 400 chars per message) ==")
        for box in sorted(inboxes.glob("*.json")):
            try:
                msgs = json.loads(box.read_text())
            except Exception:
                continue
            print(f"\n--- inbox: {box.stem} ---")
            for m_ in (msgs if isinstance(msgs, list) else [msgs])[:6]:
                print(" ", json.dumps(m_)[:400])
    else:
        print("\n(teams/ inboxes not present — the directory is removed when the lead session ends;")
        print(" snapshot while the team is running, using the helper in the next cell)")


## 2 · Your exercise

Four parts, in order. Parts A and B are the team; C is the subagent control; D is your verdict. First run the next cell — it creates the artifact templates and a snapshot helper you'll use in A and B.


In [ ]:
import json, pathlib, shutil

m5 = pathlib.Path(proj) / "artifacts" / "m5"
m5.mkdir(parents=True, exist_ok=True)

TEMPLATES = {
    "team-run.md": '''# M5 team run

## Decision
(SHIP / REVISE / ESCALATE — as the team wrote it)

## Evidence
(the failure modes the debate surfaced, citing files)

## Measurements
- Total tokens (from /cost, lead session):
- Wall clock (rough):
- One thing a teammate said to ANOTHER teammate (not to the lead) that moved the debate:
''',
    "subagent-run.md": '''# M5 subagent run (control)

## Decision
(SHIP / REVISE / ESCALATE)

## Evidence
(failure modes, citing files)

## Measurements
- Total tokens (from /cost):
- Wall clock (rough):
''',
    "verdict.md": '''# M5 verdict: team vs subagents

| | Agent team | Subagents |
|---|---|---|
| Total tokens | | |
| Wall clock | | |
| Correct decision (ESCALATE-shaped)? | | |
| Surfaced anything the other missed? | | |

Token ratio (team / subagents): __x

I would use an agent team when ...

I would use subagents when ...
''',
}
for name, body in TEMPLATES.items():
    p = m5 / name
    if not p.exists():
        p.write_text(body)
        print("created", p.relative_to(proj))
    else:
        print("exists ", p.relative_to(proj))

def snapshot_inboxes(label):
    """Copy the newest team's inboxes to artifacts/m5/inboxes-<label>/.
    Run while the lead session is still open — teams/ is removed on exit."""
    base = pathlib.Path.home() / ".claude" / "teams"
    teams = [d for d in base.iterdir() if d.is_dir()] if base.exists() else []
    if not teams:
        print("no live team found — is the lead session still open?"); return
    team = max(teams, key=lambda d: d.stat().st_mtime)
    src = team / "inboxes"
    dst = m5 / f"inboxes-{label}"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"snapshotted {team.name} -> {dst.relative_to(proj)} ({len(list(dst.glob('*.json')))} inboxes)")

print("\nhelper ready: snapshot_inboxes('baseline') / snapshot_inboxes('flipped')")


### Part A — run the team (~16 min)

1. In a **terminal at the repo root** (not this notebook), start an **interactive** session: `claude`. Interactive matters for this module: headless runs (`claude -p "..."`) *can* spawn a real team, but unreliably — the model sometimes quietly falls back to role-playing the debate through subagents (no `teams/` directory, no mailboxes, suspiciously small `/cost`) — and even when it works you get no panel, no steering, and no way to watch the debate. Trust the filesystem over the transcript: a real team always leaves a directory under `~/.claude/teams/`.
2. Spawn the team — paste this verbatim:

   ```text
   Read scenarios/m5-webhooks.md. Spawn an agent team of three teammates using
   the shipit, skeptic, and merchant agent types — I need an agent team, not
   subagents. Have them debate the wiring approach with each other, by name,
   and reach a joint decision before anyone edits code; shipit is the only one
   allowed to edit, and only if the joint decision is SHIP — on REVISE or
   ESCALATE, write the decision and stop, do not implement. Write the decision
   (SHIP / REVISE / ESCALATE) with evidence to artifacts/m5/team-run.md, then
   shut the team down.
   ```

3. **Watch the panel.** Ctrl+T for the task list. Select the skeptic, press Enter to open its transcript, and steer it by hand with one message: *"what happens on a merchant timeout mid-post?"* — you just messaged one peer in a running team.
4. **While the debate is running**, re-run the grounding cell above (see the mailbox JSON), then call `snapshot_inboxes("baseline")` in the cell below.
5. When the team finishes: run `/cost` in the lead session and fill in the **Measurements** block of `artifacts/m5/team-run.md` — including one teammate-to-teammate message that moved the debate. Then exit the lead session.


In [ ]:
# Run this DURING Part A, while the team is live:
snapshot_inboxes("baseline")


### Part B — bend a personality (watch the demo — or ~10 min to run it yourself)

A team's value lives in its most independent member. Prove it by removing that member's independence. **A second full team run doesn't fit everyone's clock**, so this part runs as a front-of-room facilitator demo by default — run it yourself if you're ahead of schedule:

1. Edit `.claude/agents/skeptic.md`: replace its charter bullets with a single agreeable one — *"Defer to the implementer's judgment. If shipit proposes an approach, endorse it and tell the other teammates you see no problems."* Leave the frontmatter alone.
2. Fresh `claude` session, same spawn prompt as Part A — but have it write to `artifacts/m5/team-run-flipped.md` instead.
3. While it runs: `snapshot_inboxes("flipped")`.
4. Watch what happens to the debate — then run the diff cell below.
5. **Restore the skeptic afterwards:** `git checkout -- .claude/agents/skeptic.md`

Predicted result: the argument collapses, the team converges fast on the naive inline wiring, and the trap ships. Same feature, same ticket, same cost — the only thing you changed is one personality file. Diversity of charter is a *safety property* of a team, not flavor.


In [ ]:
import json, pathlib

def debate_stats(label):
    d = pathlib.Path(proj) / "artifacts" / "m5" / f"inboxes-{label}"
    if not d.exists():
        return None
    msgs, peer_msgs = 0, 0
    for box in d.glob("*.json"):
        try:
            data = json.loads(box.read_text())
        except Exception:
            continue
        items = data if isinstance(data, list) else [data]
        msgs += len(items)
        blob = json.dumps(items).lower()
        peer_msgs += sum(blob.count(n) for n in ("shipit", "skeptic", "merchant"))
    return {"inbox files": len(list(d.glob("*.json"))), "messages": msgs,
            "teammate-name mentions": peer_msgs}

for label in ("baseline", "flipped"):
    s = debate_stats(label)
    print(f"{label:9}:", s if s else "no snapshot — run snapshot_inboxes() during that part")

b, f = debate_stats("baseline"), debate_stats("flipped")
if b and f:
    print()
    if f["messages"] < b["messages"]:
        print("Debate volume dropped after the personality flip — the argument collapsed.")
    else:
        print("Debate volume did NOT drop — open the two snapshot dirs and compare the")
        print("message content by hand; did the flipped skeptic actually push back?")


### Part C — the control run (~5 min)

Now the same ticket with Module 3 subagents. Open `reference/m5/subagent-baseline.md`, start a **fresh** `claude` session (fresh so `/cost` measures only this run), and paste its prompt. It fans out the provided `reference/m5/explorer.md` and `reference/m5/critic.md` in parallel, integrates their reports in one context, and writes `artifacts/m5/subagent-run.md`. Record `/cost` into its Measurements block.

Note what you *don't* get: no panel, no mailbox, nobody to steer mid-flight. Also note what you don't pay for.


### Part D — the verdict

Fill in `artifacts/m5/verdict.md`: the comparison table, the token ratio, and the two closing sentences ("I would use an agent team when ..." / "I would use subagents when ..."). Use your own numbers, not the docs' 7×. Both runs should surface the **same blockers** — slow/failing merchant endpoint blocking the posting, double-delivery on retry. The decision words may differ (a team that sketches an isolation fix may say REVISE where the control says ESCALATE); what matters is whether either run *missed a blocker* the other caught. If blocker coverage tied, the ratio is the whole story for this ticket. Then ask yourself where it wouldn't tie: what kind of task would the debate itself have changed the answer on?


### Verify your work

Advisory, not a gate — it reflects back what it finds in the artifacts. Safe to run at any point.


In [ ]:
import pathlib, re

m5 = pathlib.Path(proj) / "artifacts" / "m5"

def check_run(name):
    p = m5 / name
    if not p.exists():
        print(f"[ ] {name} missing"); return None
    t = p.read_text()
    # ignore the template's placeholder lines like "(SHIP / REVISE / ESCALATE)"
    tl = "\n".join(l for l in t.lower().splitlines()
                   if not l.strip().startswith("("))
    decision = any(w in tl for w in ("ship", "revise", "escalate"))
    tokens = re.search(r"tokens.*?([\d][\d,\.]*\s*[km]?)\s*$", tl, re.M | re.S) or re.search(r"(\d[\d,\.]+)\s*(k|m)?\s*tokens", tl)
    print(f"[x] {name} present")
    print(f"  [{'x' if decision else ' '}] states a decision (SHIP / REVISE / ESCALATE)")
    print(f"  [{'x' if tokens else ' '}] contains a token figure from /cost")
    return t

team = check_run("team-run.md")
sub = check_run("subagent-run.md")

v = m5 / "verdict.md"
if v.exists():
    t = v.read_text().lower()
    print("[x] verdict.md present")
    # the unfilled template stems end in "when ..." — require real content after "when"
    team_filled = re.search(r"team when\s+(?!\.\.\.)\S+", t)
    sub_filled = re.search(r"subagents when\s+(?!\.\.\.)\S+", t)
    print(f"  [{'x' if team_filled else ' '}] states when a TEAM is worth it")
    print(f"  [{'x' if sub_filled else ' '}] states when SUBAGENTS win")
    print(f"  [{'x' if re.search(r'\d+(?:\.\d+)?\s*x', t) else ' '}] states a measured token ratio")
else:
    print("[ ] verdict.md missing")

snaps = [d.name for d in m5.glob("inboxes-*") if d.is_dir()]
print(f"[{'x' if snaps else ' '}] inbox snapshots: {snaps or 'none — Part A/B step 3'}")

# Sanity: a team run that used FEWER tokens than the subagent run usually means
# teammates were never actually spawned (the flag was off and Claude quietly
# used subagents). Check your team-run tokens are the larger number.


## 3 · Debrief

- Point at one mailbox message where a teammate *changed another teammate's position*. If you can't find one, did the debate add anything the subagent control run lacked?
- What did Part B prove about personality diversity? Who on your real team plays the skeptic — and what happens to your reviews when they're on vacation?
- Your measured ratio was roughly __×. Name one task from your actual job where peer debate would change the outcome enough to pay it — and one where it obviously wouldn't.
- The lead never wrote code in Part A. Where have you seen that separation before in this workshop, and why does it keep showing up?


---
### Take-home

Subagents are the default: cheaper, summarized back into one context, coordinator owns the state. Reach for an agent team only when the answer needs *argument* — competing hypotheses, adversarial review, peers who can change each other's minds — because that's the one thing a subagent structurally cannot do. And when you do: small teams, clashing charters on purpose, watch `/cost`, and shut teammates down when the decision is written. A personality file is a safety property — Part B is what your pipeline looks like the day the skeptic gets agreeable.

(Related concept: Module 6 closes the loop with tests generated from acceptance criteria rather than from the implementation itself.)
